# Geico Quote Decision Predictor

---

## 1 · Problem Overview
The objective of this project is to predict whether or not a customer will approve or deny an insurance quote using machine learning techniques. I will execute an end-to-end analytical approach and apply machine learning models to internal company sales data to help the business gain a better understanding of the patterns and data dimensions demystifying customer needs and respond more strategically through informed and evaluated business recommendations.

---
## 2 · Data Collection & Cleaning

Load the proprietary quote dataset, then clean:
- Rename columns for clarity (snake_case)
- Check for missing values
- Apply data transformations:
  - Unit normalization to mitigate outliers
  - Log transformation to reduce skewness in price and features  
- Detect and address outliers (IQR)

#### 2.1 · Imports

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -----------------------------
# Add project root to sys.path for module imports
# -----------------------------
root = "/Users/phillipsmith/Desktop/Python/quote_decision_predictor"
sys.path.append(root)

# -----------------------------
# Import external python modules
# -----------------------------
from pathlib import Path

from src.preprocessing import data_preparation, data_transformation, feature_selection
from src.viz import eda
from src.models import classifiers

#### 2.2 · Configuration & Data Loading

In [2]:
# -----------------------------
# Display full dataframes
# -----------------------------
pd.set_option('display.max_rows', None) # reset_option to compact dataframe view
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

# -----------------------------
# Define path to raw Excel data source
# -----------------------------
excel_path = Path(
    '/Users/phillipsmith/Desktop/Python/quote_decision_predictor/src/data/raw/P1_Geico_Quote_Data.xlsx'
)

# -----------------------------
# Read raw data from Excel source
# -----------------------------
df_raw = pd.read_excel(excel_path, skiprows=0)
print(f"Raw DataFrame shape: {df_raw.shape}")

df_raw.columns

Raw DataFrame shape: (5000, 17)


Index(['QuoteId', 'ZipCode', 'QuoteType', 'QuoteSts', 'PHolderAge',
       'PHolderGender', 'PHolderMSts', 'PHolderHomeSts', 'PHolderDriveExp',
       'PHolderEdu', 'PHolderDriveRisk', 'CarAge', 'CarType', 'OwnStatus',
       'DailyAvgMiles', 'AnnualMileageRange', 'NoOfVehicles'],
      dtype='object')

#### 2.3 · Data Cleaning

Column headers are not standardized, so I need to rename them for clarity.

In [3]:
df_cleaned = data_preparation.clean_data(df_raw)

# # statistical summary
df_cleaned.describe()

,ID,ZIP,AGE_PH
count,5000.000000,5000.000000,5000.000000
mean,2500.500000,102083.950000,26.464000
std,1443.520003,4874640.499422,11.599621
min,1.000000,6375.000000,18.000000
25%,1250.750000,33033.000000,21.000000
50%,2500.500000,33147.000000,23.000000
75%,3750.250000,33175.000000,26.000000
max,5000.000000,344722250.000000,103.000000


#### 2.4 · Missing Values
Check for nulls. Apply imputation as needed (mean/median for numeric, mode for categorical).

In [4]:
print(df_cleaned[df_cleaned.columns].isna().sum()) # None

ID                0
ZIP               0
TYPE_Q            0
STAT_Q            0
AGE_PH            0
SEX               0
MARR_STAT         0
HOME_STAT         0
DRIVE_XP          0
EDU_PH            0
DRIVE_RISK        0
AGE_CAR           0
TYPE_CAR          0
OWN_CAR           0
AVG_MILE_DAILY    0
ANNUAL_MILE       0
NUM_CAR           0
dtype: int64


No missing values found in the dataframe.

#### 2.5 · Data Transformations

Apply unit normalization, log transforms, outlier detection, and one hot encoding to numeric features to account for outliers, reduce skewness, and improve feature quality of independent and dependent variables.

In [5]:
df_transformed = data_transformation.transform_data(df_cleaned.copy())
df_transformed.head()

,ID,ZIP,TYPE_Q,STAT_Q,AGE_PH,SEX,HOME_STAT,DRIVE_XP,EDU_PH,DRIVE_RISK,AGE_CAR,OWN_CAR,AVG_MILE_DAILY,ANNUAL_MILE,NUM_CAR,MARR_STAT_Divorced,MARR_STAT_Married,MARR_STAT_Single,TYPE_CAR_Luxury,TYPE_CAR_Minivan,TYPE_CAR_SUV,TYPE_CAR_Sedan,TYPE_CAR_Sports
0,1,32940,0,1,21,1,0,0,1,1,2,1,3,3,3,0,0,1,0,0,0,0,1
1,2,33125,1,1,37,0,1,1,2,4,1,0,3,3,2,0,0,1,0,1,0,0,0
2,3,33147,1,1,37,0,0,1,1,2,4,1,3,3,2,0,1,0,0,0,1,0,0
3,4,33305,1,1,30,0,0,0,1,2,2,1,3,3,2,0,0,1,0,0,1,0,0
4,5,33165,1,1,20,0,1,0,0,1,1,0,4,4,1,0,1,0,0,0,0,1,0


---
## 3 · Exploratory Data Analysis (EDA)

#### 3.1 · Generate EDA Plots
- Generate histograms, count plots, and a correlation matrix for numeric features (more info on bin parameters: 
(https://numpy.org/doc/stable/reference/generated/numpy.histogram_bin_edges.html#numpy.histogram_bin_edges)
- Look for skewness → candidates for log and Box-Cox transforms
    - bins='fd' (Freedman-Diaconis) is often good for skewed data
- Plots are exported to (`docs`) directory for review.

In [6]:
"""HISTOGRAM, COUNT, SCATTER PLOTS, & CORRELATION MATRIX"""

df_eda = eda.explore_data(df_transformed)

While correlation is a useful first-pass filter, I will also consider feature selection methods to capture non-linear relationships between features that correlation may miss.

---
## 4 · Feature Selection

- Determine importance of features by applying feature selection methods
- Use error metrics (AUC, Precision, Recall, F1, etc) to evaluate feature subsets and select the optimal set for modeling

#### 4.0 · Prepare Feature Set
Drop `ZIP` before feature selection. It is a high-cardinality identifier, not a categorical predictor — feeding it to a chi-square test produces an artificially huge statistic that reflects its magnitude, not any real association with the target.

In [7]:
# Drop ZIP: high-cardinality identifier, not a valid chi-square feature
df_eda = df_eda.drop(columns=['ZIP'], errors='ignore')

df_eda.head()

,TYPE_Q,STAT_Q,AGE_PH,SEX,HOME_STAT,EDU_PH,DRIVE_RISK,AGE_CAR,OWN_CAR,AVG_MILE_DAILY,NUM_CAR,MARR_STAT_Divorced,MARR_STAT_Married,MARR_STAT_Single,TYPE_CAR_Luxury,TYPE_CAR_Minivan,TYPE_CAR_SUV,TYPE_CAR_Sedan,TYPE_CAR_Sports
0,0,1,21,1,0,1,1,2,1,3,3,0,0,1,0,0,0,0,1
1,1,1,37,0,1,2,4,1,0,3,2,0,0,1,0,1,0,0,0
2,1,1,37,0,0,1,2,4,1,3,2,0,1,0,0,0,1,0,0
3,1,1,30,0,0,1,2,2,1,3,2,0,0,1,0,0,1,0,0
4,1,1,20,0,1,0,1,1,0,4,1,0,1,0,0,0,0,1,0


#### 4.1 · Chi-Square Test
Score each feature against the target (`STAT_Q`) with the chi-square test and keep those significantly associated (p < 0.05). Returns a modeling-ready DataFrame of the selected features plus the target.

In [8]:
df_features = feature_selection.FeatureSelection(df_eda).run()

df_features.head()

      Chi-square scores (sorted by statistic):
           feature       chi2  p_value
            TYPE_Q 176.277314 0.000000
               SEX 162.823422 0.000000
        DRIVE_RISK   8.403812 0.003744
   TYPE_CAR_Luxury   6.175180 0.012955
    AVG_MILE_DAILY   4.494275 0.034009
            AGE_PH   4.471400 0.034467
  TYPE_CAR_Minivan   2.699700 0.100367
   TYPE_CAR_Sports   1.553534 0.212614
    TYPE_CAR_Sedan   1.087288 0.297073
MARR_STAT_Divorced   0.722116 0.395451
           OWN_CAR   0.510972 0.474718
            EDU_PH   0.466317 0.494686
           AGE_CAR   0.084349 0.771488
           NUM_CAR   0.060315 0.805999
  MARR_STAT_Single   0.046636 0.829023
 MARR_STAT_Married   0.038510 0.844423
      TYPE_CAR_SUV   0.024029 0.876812
         HOME_STAT   0.001901 0.965222

      Selected 6 feature(s) at alpha=0.05:
      ['TYPE_Q', 'SEX', 'DRIVE_RISK', 'TYPE_CAR_Luxury', 'AVG_MILE_DAILY', 'AGE_PH']


,TYPE_Q,SEX,DRIVE_RISK,TYPE_CAR_Luxury,AVG_MILE_DAILY,AGE_PH,STAT_Q
0,0,1,1,0,3,21,1
1,1,0,4,0,3,37,1
2,1,0,2,0,3,37,1
3,1,0,2,0,3,30,1
4,1,0,1,0,4,20,1


#### 4.5 DataFrame Modeling
Selected independent variables (X) and a dependent variable (y). 

Display summary statistics for the final feature set.

In [9]:
df_modeled = df_features

df_modeled.describe()

,TYPE_Q,SEX,DRIVE_RISK,TYPE_CAR_Luxury,AVG_MILE_DAILY,AGE_PH,STAT_Q
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.586000,0.451600,1.998400,0.098400,2.019800,26.464000,0.685800
std,0.492598,0.497702,1.237539,0.297885,1.146678,11.599621,0.464243
min,0.000000,0.000000,0.000000,0.000000,0.000000,18.000000,0.000000
25%,0.000000,0.000000,1.000000,0.000000,1.000000,21.000000,0.000000
50%,1.000000,0.000000,2.000000,0.000000,2.000000,23.000000,1.000000
75%,1.000000,1.000000,3.000000,0.000000,3.000000,26.000000,1.000000
max,1.000000,1.000000,4.000000,1.000000,4.000000,103.000000,1.000000


---
## 5 · Data Splitting

#### 5.1 · Train/Test Split
Split data into training and testing sets. Start with 80/20, then experiment with 50/50 and 95/5 splits.

Each model class performs a stratified train/test split internally, e.g. `classifiers.LogisticRegressionModel(test_size=0.2, random_state=42)`. Stratification preserves the ~69/31 approve/deny class balance in both splits.

#### 5.2 · Strategy
- **Multiple split ratios** — 80/20 (default), 50/50 (stress test), 95/5 (data-hungry models)
- **Stratified sampling** — stratify on the target (`STAT_Q`) so each split preserves the class balance

---
## 6 · Baseline Model

#### 6.1 · Logistic Regression
Train logistic regression as the baseline classifier. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and the confusion matrix.

In [10]:
# LOGISTIC REGRESSION (80/20 train-test-split)

X = df_modeled.drop(columns=['STAT_Q'])
y = df_modeled['STAT_Q']

logistic_regression = classifiers.LogisticRegressionModel(test_size=0.2, random_state=42)
logistic_regression.fit_predict(X, y)
logistic_regression.evaluate()


      LOGISTIC_REGRESSION
        accuracy: 0.7810
        precision: 0.8013
        recall: 0.9052
        f1: 0.8501
        roc_auc: 0.7026
        confusion_matrix:
[[160 154]
 [ 65 621]]


{'model': 'logistic_regression',
 'accuracy': 0.781,
 'precision': 0.8012903225806451,
 'recall': 0.9052478134110787,
 'f1': 0.8501026694045175,
 'roc_auc': 0.7025635549943363}

---
## 7 · Ensemble Models

#### 7.1 · Random Forest (Bagging)
Train a Random Forest classifier — a bagging ensemble that reduces variance by averaging many decorrelated trees. Compare against the logistic baseline.

In [11]:
# RANDOM FOREST (80/20 train-test-split)

X = df_modeled.drop(columns=['STAT_Q'])
y = df_modeled['STAT_Q']

random_forest = classifiers.RandomForestModel(test_size=0.2, random_state=42)
random_forest.fit_predict(X, y)
random_forest.evaluate()


      RANDOM_FOREST
        accuracy: 0.7300
        precision: 0.7781
        recall: 0.8484
        f1: 0.8117
        roc_auc: 0.6799
        confusion_matrix:
[[148 166]
 [104 582]]


{'model': 'random_forest',
 'accuracy': 0.73,
 'precision': 0.7780748663101604,
 'recall': 0.8483965014577259,
 'f1': 0.8117154811715481,
 'roc_auc': 0.6798782752409426}

#### 7.2 · Gradient Boosting (Boosting)
Train a Gradient Boosting classifier — a boosting ensemble that reduces bias by fitting trees sequentially on the residual errors of prior trees.

In [12]:
# GRADIENT BOOSTING (80/20 train-test-split)

X = df_modeled.drop(columns=['STAT_Q'])
y = df_modeled['STAT_Q']

gradient_boosting = classifiers.GradientBoostingModel(test_size=0.2, random_state=42)
gradient_boosting.fit_predict(X, y)
gradient_boosting.evaluate()


      GRADIENT_BOOSTING
        accuracy: 0.7850
        precision: 0.8023
        recall: 0.9111
        f1: 0.8532
        roc_auc: 0.7248
        confusion_matrix:
[[160 154]
 [ 61 625]]


{'model': 'gradient_boosting',
 'accuracy': 0.785,
 'precision': 0.8023106546854942,
 'recall': 0.9110787172011662,
 'f1': 0.8532423208191127,
 'roc_auc': 0.7248170878906612}

#### 7.3 · Model Comparison & ROC
Run all three classifiers on the same 80/20 split, compare their metrics, and plot the combined ROC curves (saved to `docs/model_plots/roc_curves.png`).

In [13]:
# MODEL COMPARISON (logistic regression, random forest, gradient boosting)

results = classifiers.run_models(df_modeled, verbose=False)
results

,model,accuracy,precision,recall,f1,roc_auc
0,gradient_boosting,0.785000,0.802311,0.911079,0.853242,0.724817
1,logistic_regression,0.781000,0.801290,0.905248,0.850103,0.702564
2,random_forest,0.730000,0.778075,0.848397,0.811715,0.679878


---
## 8 · Model Evaluation

| Metric | Business Meaning |
|---|---|
| **Accuracy** | Overall share of quote decisions predicted correctly |
| **Precision** | Of quotes predicted "approve," how many were truly approved — limits false approvals |
| **Recall** | Of truly approved quotes, how many were caught — limits missed approvals |
| **F1** | Balance of precision and recall under class imbalance |
| **ROC-AUC** | Ranking quality across all thresholds — overall discriminative power |

#### 8.1 · Model Comparison & Back-Testing

Back-testing across split ratios is the next step: re-run each model at 50/50 and 95/5 to observe how training-set size affects performance (a bias-variance check).

Example of 50/50 split --> `classifiers.RandomForestModel(test_size=0.5, random_state=42)`

Example of 95/5 split --> `classifiers.RandomForestModel(test_size=0.05, random_state=42)`